# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mariamemad975/FlyRank_ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis: One row = one content_hash_id representing a 30-day snapshot, including imp_last30, prev30, clk_last30, pos_last30, and ctr_last30. This snapshot is used to identify pages whose position is worse than the panel median (pos_last30 > 81.7).

Development Window: month = 2026-03 (mid-panel).

Sealed Test Window: 2026-06 (_sample table), used only to validate the mechanics and not for defining the label logic.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata
from huggingface_hub import hf_hub_download

hf_token = userdata.get("hf_token")

file_path = "fact_content_daily_performance/month=2026-03/data_0.parquet"

local_parquet_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename=file_path,
    repo_type="dataset",
    token=hf_token
)
connect = duckdb.connect()

df = connect.execute(
    f"SELECT * FROM '{local_parquet_path}' LIMIT 5"
).df()
print("Connection successful!")
display(df)

Connection successful!


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [14]:
feature_df = connect.execute(f"""
    SELECT
        content_hash_id,
        -- Feature 1: Average Google Search Console position
        AVG(gsc_avg_position) AS avg_gsc_position,

        -- Feature 2: Total Google Search Console impressions
        COALESCE(SUM(gsc_impressions), 0) AS total_gsc_impressions,

        -- Feature 3: Total Google Search Console clicks
        COALESCE(SUM(gsc_clicks), 0) AS total_gsc_clicks,

        -- Feature 4: Total Google Analytics 4 pageviews
        COALESCE(SUM(ga4_pageviews), 0) AS total_ga4_pageviews,

        -- Feature 5: Total Google Analytics 4 sessions
        COALESCE(SUM(ga4_sessions), 0) AS total_ga4_sessions

    FROM '{local_parquet_path}'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

display(feature_df.head())

,content_hash_id,avg_gsc_position,total_gsc_impressions,total_gsc_clicks,total_ga4_pageviews,total_ga4_sessions
0,content_7a105f548d9c6916,7.209549,6523.0,7.0,1.0,1.0
1,content_a3ea9792f793ec72,2.987198,453.0,0.0,0.0,0.0
2,content_36c36abc7650d7af,6.724039,5630.0,6.0,6.0,3.0
3,content_a7da352b73b02668,7.244844,4944.0,13.0,2.0,2.0
4,content_1855a661b4d36130,4.209227,429.0,1.0,2.0,2.0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_check = connect.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(
            DISTINCT CONCAT(
                content_hash_id,
                '||',
                CAST(report_date AS STRING))) AS unique_grain_rows
    FROM '{local_parquet_path}'
""").df()
display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_grain_rows
0,9841378,9841378


In [16]:
date_check = connect.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM '{local_parquet_path}'
""").df()

display(date_check)

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [17]:
gsc_availability = connect.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT_IF(gsc_data_available IS TRUE) AS available_rows,
        ROUND(
            COUNT_IF(gsc_data_available IS TRUE) * 100.0 / COUNT(*),2) AS availability_pct FROM '{local_parquet_path}'""").df()

display(gsc_availability)

,total_rows,available_rows,availability_pct
0,9841378,3611061.0,36.69


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Directional correlations only, not causation: Low query diversity does not necessarily cause ranking declines; it is used by the model only as a predictive signal.
Limited historical data: Missing imp_prev30 data restricts deeper historical pattern analysis beyond the available 60-day period.
Class imbalance in recall: Class 0 (non-decay) has weaker recall at 67%, compared with 97% recall for Class 1 (decay).

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.